In [ ]:
from itertools import combinations
import scipy.stats as stats
from sklearn.cluster import KMeans
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
dataset = pd.read_excel(r"E:\AI Projects\My Project\Files\aggr.xlsx", sheet_name="Golpakhsh")

In [ ]:
from langchain_openai.chat_models import ChatOpenAI
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="not-needed",
    model="qwen2.5-3b-instruct",
    temperature=0,
    max_completion_tokens=1000
)

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

x = dataset.iloc[:, 2:-1].values
y = dataset.iloc[:, -1].values
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0) 

#MLR
regressor = LinearRegression()
regressor.fit(x_train, y_train)
y_pred = regressor.predict(x_test)
np.set_printoptions(precision=2)

#DTR
regressor2 = DecisionTreeRegressor(random_state=0)
regressor2.fit(x, y)
y_pred2 = regressor2.predict(x_test)


MLR_Score = r2_score(y_test, y_pred)
DTR_Score = r2_score(y_test, y_pred2)
print("MLR_Score: ", MLR_Score)
print("DTR_Score: ", DTR_Score)

In [ ]:
def llm_ANOVA_result(stat, pv):
    """
    Uses an LLM (Agent) to explain model results in simple language.
    """
    prompt = f"""
    You are a helpful data science assistant.
    Explain the following ANOVA results in very simple terms
    without any extra information about ANOVA definition
    so that a non-technical person can understand.

    Consider the following parameters as the result of ANOVA:
    - statistics: {stat}
    - p-value: {pv}

    The goal is to compare companies sale amount.
    the final result sjows as a table. if the table output has more than one row, please anylysis row by row
    Just give the Conclusion as result.

    """

    return llm.invoke(prompt).content

In [ ]:
def llm_KMeans_result():
    """
    Uses an LLM (Agent) to explain model results in simple language.
    """
    prompt = f"""
    You are a helpful data science assistant.
    Explain the following KMeans results in very simple terms.
    without any extra information about KMeans definition
    so that a non-technical person can understand.
    The goal is to analysis the plot.

    """

    return llm.invoke(prompt).content

In [ ]:
def ANOVA_result():
    group_column = "CompanyName"
    value_column = "NetAmount"
    #groups = [group[value_column].dropna().values for c, group in dataset.groupby(group_column)]
    com = []
    for i,j in dataset.groupby(group_column):
        com.append(i)

    result = []
    for i,j in combinations(com, 2):
        group1 = dataset[dataset['CompanyName'] == i][value_column]
        group2 = dataset[dataset['CompanyName'] == j][value_column]

        statistic, pvalue = stats.f_oneway(group1, group2)
        result.append({
            'Company 1': i,
            'Company 2': j,
            'F-statistic': statistic,
            'p-value': format(pvalue, ".5f"),
            'Significant (α=0.05)': pvalue < 0.05
        })
    df = pd.DataFrame(result)
    final_df = df.where(~df['Significant (α=0.05)']).dropna()

    explanation = llm_ANOVA_result(final_df['F-statistic'], final_df['p-value'])

    return {
        'Company_1': final_df['Company 1'],
        'Company_2': final_df['Company 2'],
        'AI_Message': explanation,
        'dataset': final_df
    }

In [ ]:
def ML_Kmeans():

    x = pd.concat([dataset['NetQTY'], dataset['NetAmount']], axis=1) 
    x = np.array(x)

    sc = StandardScaler()
    x = sc.fit_transform(x)
    wcss = []
    for i in range(1, 11):
        kmeans = KMeans(n_clusters = i, init = 'k-means++', random_state= 42)
        kmeans.fit(x)
        wcss.append(kmeans.inertia_)

    df = []
    for i,j in enumerate(wcss, start=1):
        df.append([i, f"{j:.0f}"])

    df = pd.DataFrame(df, columns=["Index", "Number"])

    kmeans = KMeans(n_clusters = 4, init = 'k-means++', random_state= 42)
    y_kmeans = kmeans.fit_predict(x)
    
    fig, ax = plt.subplots()
    ax.scatter(x[y_kmeans == 0, 0], x[y_kmeans == 0, 1], s = 100, c = 'red', label = 'cluster1')
    ax.scatter(x[y_kmeans == 1, 0], x[y_kmeans == 1, 1], s = 100, c = 'blue', label = 'cluster2')
    ax.scatter(x[y_kmeans == 2, 0], x[y_kmeans == 2, 1], s = 100, c = 'green', label = 'cluster3')
    ax.scatter(x[y_kmeans == 3, 0], x[y_kmeans == 3, 1], s = 100, c = 'cyan', label = 'cluster4')
    ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], s = 300, c = 'yellow', label = 'centroids')
    ax.legend()
    #ax.show()

    explanation = llm_KMeans_result()
    return {
        'Cluster_selection': df,
        'plot': plt.show(fig),
        'AI Message': explanation
    }

In [ ]:
if __name__ == "__main__":
    result1 = ANOVA_result()
    result2 = ML_Kmeans()

    print("=========ANOVA Analysis=========")
    print(result1["dataset"])
    print(result1["AI_Message"])
    
    print("=========KMeans Analysis=========")
    print(result2["Cluster_selection"])
    print(result2["plot"])